# 研究与工程思维 4/6：不确定性、Bootstrap 与配对比较

这不是一节“记术语”的课，而是一节**改变判断过程**的实验课。

| 项目 | 内容 |
|---|---|
| 核心问题 | WER 从 10.0% 降到 9.7%，这是改进还是抽样波动？ |
| 迁移价值 | 适用于模型比较、线上指标、个人学习测验和任何有限样本结论。 |
| 建议投入 | 90～150 分钟；先预测，再运行，再保留被推翻的判断 |
| 通关证据 | 能把结论写成“主张—证据—反证—边界—下一步” |

固定闭环：

```text
观察（发生了什么） → 假设（可能为什么） → 区分性预测 → 最小实验
        ↑                                      ↓
        └──── 更新置信度、记录反例、决定下一步 ────┘
```

**观察不是原因，总分不是解释，相关不是干预效果，运行成功不是结论成立。**


## 课前预测：先暴露自己的判断规则

1. 用一句话回答：WER 从 10.0% 降到 9.7%，这是改进还是抽样波动？
2. 写出你最可能犯的判断错误，例如“只看平均值”或“看到相关就认定因果”。
3. 为本课写一个可被数据推翻的预测；不要写“应该会更好”这种没有阈值的话。
4. 写出什么结果会让你改变主意。

完成实验后回来修正。保留原答案，因为“怎样改主意”本身就是思维能力证据。


## 一手资料与课程取舍

- [SciPy：Bootstrap 置信区间](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.bootstrap.html)
- [NIST SCTK：标准 ASR 评分与置信报告](https://github.com/usnistgov/SCTK/blob/master/doc/sclite.htm)
- [Guo 等：现代神经网络置信度校准](https://proceedings.mlr.press/v70/guo17a.html)

课程把这些资料转成小型、确定性、可运行的 ASR 实验。示例数据用于理解方法，不代表真实产品结论。


## 1. 先明确统计单位

一句话内的词高度相关，同一说话人的多句话也相关。把 1000 个词假装成 1000 个独立样本会得到过窄区间。部署目标若是“新说话人”，应优先按说话人重采样；目标若是“这些固定说话人的新句子”，统计单位会不同。


In [ ]:
import numpy as np

rng = np.random.default_rng(42)
rows = []
for speaker in range(12):
    difficulty = rng.uniform(0.04, 0.20)
    for utterance in range(8):
        words = int(rng.integers(8, 18))
        err_a = int(rng.binomial(words, difficulty))
        # B 平均略好，但不是每个说话人/句子都赢
        err_b = int(rng.binomial(words, max(0.01, difficulty - 0.018)))
        rows.append({"speaker": speaker, "words": words, "A": err_a, "B": err_b})

def micro_wer(data, system):
    return sum(row[system] for row in data) / sum(row["words"] for row in data)

print("A WER", f"{micro_wer(rows, 'A'):.2%}")
print("B WER", f"{micro_wer(rows, 'B'):.2%}")
print("观察差值 B-A", f"{micro_wer(rows, 'B')-micro_wer(rows, 'A'):+.2%}")


## 2. 配对 Cluster Bootstrap

两个系统必须在同一重采样中使用同一批说话人，这保留配对关系。每轮有放回抽取说话人，连同其全部句子，计算 `WER_B - WER_A`。

区间跨 0 不等于“两个系统完全一样”，只表示当前数据和方法不足以排除零差异。区间窄且整体落在业务最小改进阈值之外，才更有决策力。


In [ ]:
def paired_speaker_bootstrap(data, n_resamples=4000, seed=0):
    speakers = sorted({row["speaker"] for row in data})
    by_speaker = {s: [row for row in data if row["speaker"] == s] for s in speakers}
    rng = np.random.default_rng(seed)
    differences = []
    for _ in range(n_resamples):
        sampled = rng.choice(speakers, size=len(speakers), replace=True)
        sample_rows = [row for s in sampled for row in by_speaker[int(s)]]
        differences.append(micro_wer(sample_rows, "B") - micro_wer(sample_rows, "A"))
    return np.asarray(differences)

boot = paired_speaker_bootstrap(rows)
low, high = np.quantile(boot, [0.025, 0.975])
observed = micro_wer(rows, "B") - micro_wer(rows, "A")
print(f"差值={observed:+.2%}, 95% percentile CI=[{low:+.2%}, {high:+.2%}]")
print("B 更好的 bootstrap 比例:", f"{np.mean(boot < 0):.1%}")
assert len(boot) == 4000


## 3. 统计差异与实际价值是两道门

发布规则示例：

1. 质量：95% 区间上界小于 0（B 大概率不退化）；
2. 价值：差值中位数至少改善 0.5pp；
3. 守门：关键切片不退化超过 1pp；
4. 系统：P95 延迟不超过预算。

样本巨大时 0.05pp 也可能“显著”，却不值得部署；样本很小时 2pp 也可能区间很宽，需要更多数据而不是武断宣布失败。


In [ ]:
import matplotlib.pyplot as plt

confidence = np.array([.05,.12,.18,.25,.34,.42,.48,.55,.62,.68,.73,.79,.84,.88,.91,.94,.96,.97,.98,.99])
correct = np.array([0,0,0,1,0,1,0,1,1,0,1,1,1,0,1,1,0,1,1,1])

def reliability_bins(confidence, correct, edges=np.linspace(0, 1, 6)):
    result = []
    for lo, hi in zip(edges[:-1], edges[1:]):
        mask = (confidence >= lo) & (confidence < hi if hi < 1 else confidence <= hi)
        if mask.any():
            result.append((lo, hi, int(mask.sum()), float(confidence[mask].mean()), float(correct[mask].mean())))
    return result

bins = reliability_bins(confidence, correct)
ece = sum(n/len(confidence) * abs(avg_c-acc) for _,_,n,avg_c,acc in bins)
for item in bins:
    print(f"[{item[0]:.1f},{item[1]:.1f}) n={item[2]:2d} conf={item[3]:.2f} acc={item[4]:.2f}")
print("ECE=", round(ece, 3))

plt.plot([0,1], [0,1], "k--", label="perfect calibration")
plt.scatter([x[3] for x in bins], [x[4] for x in bins], s=[20*x[2] for x in bins])
plt.xlabel("mean confidence"); plt.ylabel("empirical accuracy"); plt.legend(); plt.grid(True); plt.show()


## 4. 置信度必须回答“100 次中对多少次”

准确率和校准是不同轴：模型可以准确但过度自信，也可以准确率一般但置信度诚实。ECE 依赖分箱，不能单独作为证明；还要看 reliability diagram、NCE/Brier、关键阈值附近和分布外切片。


## 闭卷挑战

选两组同一样本上的系统输出，从空白实现按说话人配对 Bootstrap。报告点估计、95% 区间、最小有意义改进和关键切片守门；禁止只写‘显著/不显著’。

回答时强制使用下面的证据卡：

```text
主张：
证据：
最强替代解释：
什么结果会推翻主张：
适用边界：
下一步最小实验：
```


## 最小掌握门禁

- [ ] 我在运行前写了方向和数量级预测。
- [ ] 我能指出示例结论中至少一个替代解释。
- [ ] 我能从空白重写本课核心函数，并用边界输入测试。
- [ ] 我能说明“没有发现差异”和“证明没有差异”的区别。
- [ ] 我把一次被数据推翻的判断写入 `LEARNING_LOG.md`。
- [ ] 我能把本课方法迁移到一个非 ASR 问题。

下一步：第 5 课用不变量、变形测试和反事实更快定位根因。
